# 01. 법학 분야 초록 키워드·토픽 추출

Kiwi 형태소 분석 → Sentence-BERT 임베딩 → BERTopic(UMAP + HDBSCAN) → outlier reduction → KeyBERT 토픽 키워드, Gemini로 토픽 라벨링(선택).
API 키는 환경변수로 넣는다. 데이터(DBpia 대회 제공 JSON)는 포함하지 않는다.



```
# 코드로 형식 지정됨
```

## 패키지 설치 안녕하세요

In [ ]:
!pip -q install sentence-transformers bertopic umap-learn hdbscan kiwipiepy faiss-cpu ipywidgets pyarrow keybert

## 기본 설정

In [ ]:
import os, re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
from kiwipiepy import Kiwi
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from keybert import KeyBERT

import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 4)

pd.set_option("display.max_colwidth", 120)



```
# 코드로 형식 지정됨
```

## 데이터 로드

In [ ]:
DATA_PATH = "data/SSU_Datathon2025_사회과학분야_54328건 (1).json"
raw_df = pd.read_json(DATA_PATH)
df = pd.json_normalize(raw_df["NODE_LIST"])
df.head()

## EDA

In [ ]:
df.shape, df.columns.tolist()

In [ ]:
df["NODE_CLSS_02"].unique()

## 전처리 (for 키워드 분석)

In [ ]:
# 우선은 '한국어 초록', '영어 초록' 모두 없는 데이터 제외하고 진행
mask_no_kr = df["ABST_KR"].fillna("").astype(str).str.strip().eq("")
mask_no_en = df["ABST_EN"].fillna("").astype(str).str.strip().eq("")
df_abst = df[~(mask_no_kr & mask_no_en)].copy()

In [ ]:
# KR_ABST 우선, 없으면 EN_ABST 사용
df_abst["ABST"] = df_abst["ABST_KR"].fillna("").astype(str).str.strip()
mask_empty = df_abst["ABST"].eq("") | df_abst["ABST"].isna()
df_abst.loc[mask_empty, "ABST"] = df_abst.loc[mask_empty, "ABST_EN"].fillna("").astype(str).str.strip()

In [ ]:
# "PBSH" 컬럼의 YYYYMM에서 YYYY만 떼온 "YEAR" 컬럼 생성
df_abst["YEAR"] = df_abst["PBSH"].astype(str).str.slice(0, 4).astype(int)

In [ ]:
# "PBSH" 컬럼을 datatime 타입으로 변환한 "PBSH_DT" 컬럼 생성
df_abst["PBSH_DT"] = pd.to_datetime(df_abst["PBSH"].astype(str), format="%Y%m", errors="coerce")

In [ ]:
# 법학 데이터만 필터링
# "" 안에 중분류를 넣어서 각각 돌려보는 방법으로 하면 될듯....ㅎ
df_law = df_abst[df_abst["NODE_CLSS_02"].fillna("").eq("법학")].copy()
df_law = df_law[df_law["ABST"].fillna("").astype(str).str.strip().ne("")].copy()
df_law.reset_index(drop=True, inplace=True)
df_law.head()

## 맥락 임베딩 (Sentence-BERT)

In [ ]:
df_law = df_law[df_law["ABST"].fillna("").astype(str).str.strip().ne("")].copy()
df_law.reset_index(drop=True, inplace=True)

texts_raw = df_law["ABST"].astype(str).str.replace(r"\s+", " ", regex=True).tolist()
print("docs:", len(texts_raw))
print("example:", texts_raw[0][:200])

In [ ]:
tqdm.pandas()
kiwi = Kiwi()

ALLOWED_PREFIX = ("NN", "VV", "VA")

def kiwi_content_tokens(text: str):
    if text is None:
        return []
    text = str(text).strip()
    if not text:
        return []

    # 최소한의 정규화
    text = re.sub(r"\s+", " ", text)

    tokens = kiwi.tokenize(text)
    out = []
    for t in tokens:
        # 품사 기반 자동 포함
        if t.tag.startswith(ALLOWED_PREFIX):
            tok = t.form.strip()
            # 한 글자 토큰은 잡음이 많아서 제거(원하면 끄기)
            if len(tok) >= 2:
                out.append(tok)
    return out

# 빠른 동작 확인
print(kiwi_content_tokens(texts_raw[0])[:100])

In [ ]:
# Sentence-BERT 임베딩 (일단은 다국어 버전으로)
EMB_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

embedder = SentenceTransformer(EMB_MODEL_NAME, device=device)

embeddings = embedder.encode(
    texts_raw,
    batch_size=128 if device == "cuda" else 32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)

print("embeddings:", embeddings.shape)

In [ ]:
# Kiwi 토큰화는 여기서 딱 1번만 수행 (매우 중요)
df_law["TOK_STR"] = df_law["ABST"].progress_apply(lambda x: " ".join(kiwi_content_tokens(str(x))))

# 빈 토큰 제거
df_law = df_law[df_law["TOK_STR"].str.strip().ne("")].copy()
df_law.reset_index(drop=True, inplace=True)

# embeddings도 df_law와 행 수가 같아야 하니, embeddings도 다시 맞춰줌(필수)
texts_raw = df_law["ABST"].astype(str).str.replace(r"\s+", " ", regex=True).tolist()
embeddings = embedder.encode(
    texts_raw,
    batch_size=128 if device == "cuda" else 32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)

## 토픽 후보 생성 (BERTopic(UMAP, HDBSCAN, c-TF-IDF))

In [ ]:
# UMAP/HDBSCAN 파라미터는 데이터에 민감합니다.
# 아래는 "안전한 기본값" 성격. (필요하면 min_cluster_size부터 조정 권장)
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    tokenizer=str.split,  # 원문을 넣어도 내부에서 내용어 토큰화됨
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.7,   # 너무 흔한 토큰 자동 제거 (수동 불용어 대신)
)

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True  # 토픽 분별력 낮은 단어 자동 감쇠
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    language="multilingual",
    calculate_probabilities=False,
    verbose=True
)

topics, _ = topic_model.fit_transform(df_law["TOK_STR"].tolist(), embeddings=embeddings)

df_law["TOPIC_ID"] = topics
topic_info = topic_model.get_topic_info()
topic_info.head(10)

In [ ]:
# 1. Outlier Reduction 실행 (c-tf-idf 전략 사용)
# 'topics'는 기존 fit_transform에서 얻은 결과 리스트입니다.
new_topics = topic_model.reduce_outliers(
    df_law["TOK_STR"].tolist(), 
    topics, 
    strategy="c-tf-idf"
)

# 2. 기존 데이터프레임의 TOPIC_ID 업데이트
df_law["TOPIC_ID"] = new_topics

# 3. 모델의 토픽 정보를 업데이트 (이 과정을 거쳐야 topic_model의 내부 통계가 바뀝니다)
topic_model.update_topics(df_law["TOK_STR"].tolist(), topics=new_topics)

# 4. 결과 확인
new_topic_info = topic_model.get_topic_info()
print("--- Outlier Reduction 적용 후 토픽 요약 ---")
display(new_topic_info.head(10))

# 5. -1번 토픽(Outlier)이 얼마나 줄었는지 확인
outlier_count = len(df_law[df_law["TOPIC_ID"] == -1])
print(f"\n남아있는 Outlier(-1) 개수: {outlier_count}개")

## 연도별 토픽 집계

In [ ]:
# -1(아웃라이어) 제외
df_in = df_law[df_law["TOPIC_ID"] != -1].copy()

# 토픽 라벨(상위 단어로 간단 라벨)
def get_top_words(topic_id: int, topn: int = 20):
    tw = topic_model.get_topic(topic_id)
    if not tw:
        return []
    return [w for w, _ in tw[:topn]]

def topic_label_from_words(tid, k=6):
    words = get_top_words(int(tid), topn=k)
    return " / ".join(words) if words else None

topic_meta = topic_model.get_topic_info().copy()
topic_meta = topic_meta[topic_meta["Topic"] != -1].copy()
topic_meta["LABEL"] = topic_meta["Topic"].apply(lambda t: topic_label_from_words(t, k=6))

topic_meta_small = topic_meta[["Topic", "Count", "LABEL"]].rename(
    columns={"Topic": "TOPIC_ID", "Count": "TOPIC_TOTAL"}
)

# 연도별 전체 문서수(토픽 할당된 것만)
year_total = df_in.groupby("YEAR").size().rename("YEAR_TOTAL")

# 연도-토픽별 카운트/비중
year_topic = (
    df_in.groupby(["YEAR", "TOPIC_ID"])
         .size()
         .rename("N")
         .reset_index()
)
year_topic = year_topic.merge(year_total.reset_index(), on="YEAR", how="left")
year_topic["SHARE"] = year_topic["N"] / year_topic["YEAR_TOTAL"]

# 연도 내 순위
year_topic["RANK_IN_YEAR"] = (
    year_topic.groupby("YEAR")["N"]
              .rank(method="dense", ascending=False)
              .astype(int)
)

# 라벨/토픽 전체크기 붙이기
year_topic = year_topic.merge(topic_meta_small, on="TOPIC_ID", how="left")

# 연도별 TOPK 토픽 출력
TOPK = 15
year_topk = (
    year_topic.sort_values(["YEAR", "N"], ascending=[True, False])
              .groupby("YEAR")
              .head(TOPK)
              .sort_values(["YEAR", "N"], ascending=[True, False])
)

for y, sub in year_topk.groupby("YEAR"):
    print(f"\n==== {y}년 TOP {TOPK} 토픽 ====")
    for _, r in sub.iterrows():
        print(f"{int(r.RANK_IN_YEAR):2d}. T{int(r.TOPIC_ID):3d}  n={int(r.N):4d}  share={r.SHARE:.3f}  | {r.LABEL}")

# (원하면) 매트릭스도 같이 만들어두기
mat_count = year_topic.pivot_table(index="YEAR", columns="TOPIC_ID", values="N", fill_value=0)
mat_share = year_topic.pivot_table(index="YEAR", columns="TOPIC_ID", values="SHARE", fill_value=0)

display(year_topic.head())
display(mat_count.head())
display(mat_share.head())

## LLM으로 최종 토픽 생성

In [ ]:
import os
import requests
import json
from tqdm import tqdm

# 설정 변경
LLM_PROVIDER = "gemini"
# Gemini 1.5 Flash는 속도가 빠르고 비용 효율적이라 대량의 토픽 처리에 적합합니다.
GEMINI_MODEL = "gemini-2.0-flash"
# 실행 전 셸에서: export GEMINI_API_KEY=...

def build_topic_rename_prompt(node_clss_02: str, top_words: list[str], abstracts: list[str]) -> str:
    candidates = ", ".join(top_words)
    abst_block = "\n\n".join([f"- {a}" for a in abstracts])

    return f"""
당신은 사회과학 분야 중에서도 '{node_clss_02}' 중분야를 20년 이상 연구해 온 학술 연구자입니다.
당신의 역할은 토픽을 새로 만들거나 외부 지식을 추가하는 것이 아니라,
이미 주어진 정보만으로 클러스터의 공통 주제를 '정제하고 명명'하는 것입니다.

[규칙]
1) 토픽명/설명은 반드시 아래 '후보 단어' 또는 '초록'에 실제로 등장한 용어만 사용하세요.
2) 후보 단어/초록에 없는 개념·용어(학설/판례/법률명 포함)는 절대 새로 만들지 마세요.
3) '연구/분석/결과/논의/고찰' 같은 일반어는 가능하면 제외하세요.
4) 확실하지 않으면 과도한 추론 없이 보수적으로 명명하세요.

[입력]
- NODE_CLSS_02: {node_clss_02}
- 후보 단어: {candidates}

- 초록:
{abst_block}

[출력 형식]
NODE_CLSS_02: {node_clss_02}
토픽명: <5~15자 명사구>
설명:
- <2~3문장, 반드시 후보 단어/초록 내 용어만 사용>
""".strip()

def llm_rename_topics_gemini(topic_model, df_docs, node_clss_02="법학",
                             topn_words=50, sample_docs=12):
    api_key = os.environ.get("GEMINI_API_KEY", "").strip()
    if not api_key:
        print("⚠️ GEMINI_API_KEY가 없어 LLM 재명명을 스킵합니다.")
        return {}

    headers = {"Content-Type": "application/json"}
    
    # ⚠️ 수정된 엔드포인트 URL: v1beta 사용 및 모델 경로 명시
    # models/ 뒤에 모델명이 바로 오도록 구성해야 합니다.
    base_url = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent?key={api_key}"

    rename_map = {}
    topic_ids = sorted([t for t in df_docs["TOPIC_ID"].unique() if t != -1])

    for tid in tqdm(topic_ids, desc="Gemini topic renaming"):
        topic_words = topic_model.get_topic(tid)
        if not topic_words: continue
        top_words = [w for w, _ in topic_words[:topn_words]]
        
        sub = df_docs[df_docs["TOPIC_ID"] == tid]
        abst_samples = (
            sub["ABST"].astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.slice(0, 800)
            .sample(n=min(sample_docs, len(sub)), random_state=42)
            .tolist()
        )

        prompt = build_topic_rename_prompt(node_clss_02, top_words, abst_samples)

        payload = {
            "contents": [{
                "parts": [{"text": prompt}]
            }],
            "generationConfig": {
                "temperature": 0.1,
                "maxOutputTokens": 1024
            }
        }

        try:
            # 무료 티어 배려를 위한 짧은 대기
            import time
            time.sleep(1)

            r = requests.post(base_url, headers=headers, json=payload, timeout=60)
            
            if r.status_code != 200:
                print(f"\n❌ Error {r.status_code}: {r.text}")
                continue

            data = r.json()
            if "candidates" in data and data["candidates"]:
                text_out = data["candidates"][0]["content"]["parts"][0]["text"].strip()
                rename_map[tid] = text_out
                
        except Exception as e:
            print(f"⚠️ topic {tid} Gemini rename 실패:", repr(e))
            continue

    return rename_map

In [ ]:
# 실행 (수정된 부분)
rename_map = {}
if USE_LLM_RENAME:
    # ⚠️ llm_rename_topics_openai 대신 gemini 함수를 호출해야 합니다.
    rename_map = llm_rename_topics_gemini(topic_model, df_law, node_clss_02="법학")
    print("renamed topics:", len(rename_map))

def default_topic_label(topic_model, topic_id: int, topk=5) -> str:
    words = topic_model.get_topic(topic_id) or []
    words = [w for w, _ in words[:topk]]
    return " / ".join(words) if words else "N/A"

In [ ]:
# topic_info에 토픽 라벨 컬럼 추가
topic_labels = []
for tid in topic_info["Topic"].tolist():
    if tid == -1:
        topic_labels.append("OUTLIER")
    else:
        # LLM 결과가 있으면 그대로, 없으면 기본라벨
        if tid in rename_map and rename_map[tid].strip():
            topic_labels.append(rename_map[tid].strip())
        else:
            topic_labels.append(default_topic_label(topic_model, tid))

topic_info["TOPIC_LABEL"] = topic_labels
topic_info.head(10)

## KeyBERT로 후보 키워드 추출

In [ ]:
kw_model = KeyBERT(model=embedder)

def extract_topic_keywords_keybert(
    texts_for_topic,
    top_n=10,
    per_doc_top_n=10,
    keyphrase_ngram_range=(1, 2)
):
    freq = {}
    score_sum = {}

    for text in texts_for_topic:
        text = str(text).strip()
        if not text:
            continue

        kws = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=keyphrase_ngram_range,
            top_n=per_doc_top_n,
            use_mmr=False
        )

        for kw, score in kws:
            kw = str(kw).strip()
            if not kw:
                continue
            freq[kw] = freq.get(kw, 0) + 1
            score_sum[kw] = score_sum.get(kw, 0.0) + float(score)

    ranked = sorted(freq.keys(), key=lambda k: (freq[k], score_sum.get(k, 0.0)), reverse=True)
    topk = ranked[:top_n]

    return {
        "keywords": topk,
        "freq": {k: freq[k] for k in topk},
        "score_sum": {k: score_sum.get(k, 0.0) for k in topk},
    }

topic_keyword_rows = []
topic_ids = sorted([t for t in df_law["TOPIC_ID"].unique() if t != -1])

for tid in tqdm(topic_ids, desc="KeyBERT keywords per topic"):
    sub = df_law[df_law["TOPIC_ID"] == tid]
    out = extract_topic_keywords_keybert(
        sub["ABST"].tolist(),
        top_n=10,
        per_doc_top_n=10
    )

    topic_keyword_rows.append({
        "TOPIC_ID": tid,
        "DOC_COUNT": len(sub),
        "TOP_KEYWORDS": out["keywords"],
        "TOP_KEYWORDS_FREQ": out["freq"],
        "TOP_KEYWORDS_SCORE_SUM": out["score_sum"]
    })

df_topic_keywords = pd.DataFrame(topic_keyword_rows)

topic_summary = (
    topic_info.rename(columns={"Topic": "TOPIC_ID", "Count": "DOC_COUNT"})
    .query("TOPIC_ID != -1")
    .merge(df_topic_keywords, on=["TOPIC_ID", "DOC_COUNT"], how="left")
    .sort_values("DOC_COUNT", ascending=False)
    .reset_index(drop=True)
)

topic_summary.head(20)